# 05 — EDA tìm feature cho LightGBM v4 (M2 nấc 0)

**Mục tiêu:** trả lời *"Tại sao model tách được `di_ngang` (F1 0.61) nhưng kém `tang`/`giam` (0.29–0.30)?"*
→ Bài toán ±1% thực chất là dự **biên độ** trước, **hướng** sau. Tìm bằng chứng cho 3 nhóm feature ứng viên:
1. **Volatility** (tính từ `close` sẵn có — ưu tiên #1, panel hiện CHƯA có nhóm này)
2. **Market context** (VNINDEX qua vnstock — foreign flows KHÔNG khả dụng vnstock free, xem TICKLIST nấc 1)
3. **Sửa redundancy** feature hiện có (ma7 ~ ma20 ~ close → thay bằng tỷ lệ)

## Quy tắc bắt buộc
- **CHỈ dùng dữ liệu `date < 2025-01-01`** (train+val). Test 2025 giữ nguyên trinh — nhìn test khi thiết kế feature = leak gián tiếp, gate macro-F1 > 0.4018 mất ý nghĩa.
- Mọi feature ứng viên chỉ dùng thông tin **≤ 16:00 ngày T** (rolling nhìn về quá khứ; volume phiên T chốt 15:00 → hợp lệ).
- **Bẫy multiple comparison:** thử nhiều feature thì vài cái "đẹp" do may mắn. Biểu đồ/MI ở đây chỉ để CHỌN ứng viên; phán quyết cuối là walk-forward val ở nấc train v4.

## Cách chạy
```bash
cd backend && uv run --with jupyter --with scikit-learn python -m jupyter lab ../ml/notebooks/05_eda_market_features.ipynb
```
(backend env có sẵn vnstock + pandas + matplotlib + seaborn; jupyter + sklearn thêm qua `--with`)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Tìm repo root (cwd có thể là ml/notebooks hoặc backend tuỳ cách mở jupyter)
REPO = Path.cwd().resolve()
is_kaggle = False

# Kiểm tra xem có đang chạy trên Kaggle không
if Path("/kaggle").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    is_kaggle = True

if is_kaggle:
    print("Detected Kaggle environment. Checking for vnstock...")
    try:
        import vnstock  # noqa: E402, F401

        print("vnstock is already installed.")
    except ImportError:
        print("vnstock not found. Installing vnstock...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "vnstock", "--quiet"])
        print("vnstock installed successfully.")

    # Trên Kaggle, ta tìm file dataset từ đường dẫn do Kaggle cung cấp
    kaggle_dataset_path = Path("/kaggle/input/datasets/chithanhdeptrai/tradepilot-panel/data")
    if kaggle_dataset_path.is_file():
        PANEL = kaggle_dataset_path
    elif kaggle_dataset_path.is_dir():
        # Nếu là thư mục, tìm file csv
        csv_files = list(kaggle_dataset_path.glob("*.csv"))
        if csv_files:
            PANEL = csv_files[0]
        else:
            csv_files_recursive = list(kaggle_dataset_path.glob("**/*.csv"))
            if csv_files_recursive:
                PANEL = csv_files_recursive[0]
            else:
                PANEL = kaggle_dataset_path / "training_panel.csv"
    else:
        # Check if we can search in /kaggle/input for any training_panel.csv or similar
        potential_files = list(Path("/kaggle/input").glob("**/training_panel.csv"))
        if potential_files:
            PANEL = potential_files[0]
        else:
            PANEL = kaggle_dataset_path

    # Cache trên Kaggle cần ghi được, nên để ở /kaggle/working/eda_cache
    CACHE = Path("/kaggle/working/eda_cache")
else:
    # Chạy cục bộ (Local)
    while not ((REPO / "ml").is_dir() and (REPO / "backend").is_dir()):
        assert REPO != REPO.parent, "Không tìm thấy repo root (cần thư mục ml/ + backend/)"
        REPO = REPO.parent
    PANEL = REPO / "ml" / "data" / "training_panel.csv"
    CACHE = REPO / "ml" / "data" / "eda_cache"  # ml/data/ đã gitignore

EDA_END = "2025-01-01"  # CHỈ EDA trước mốc này — test 2025 giữ nguyên trinh
CACHE.mkdir(parents=True, exist_ok=True)

CLASSES = ["di_ngang", "giam", "tang"]
FEATURES = ["ma7", "ma20", "rsi14", "macd", "macd_signal", "sentiment_agg", "news_count"]
SEED = 42

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160)
print("Repo:", REPO)
print("Panel path:", PANEL)
print("Cache path:", CACHE)

## 1. Load panel + cắt ranh giới EDA

**Câu hỏi cần trả lời:** dữ liệu phủ bao nhiêu mã/năm? NaN warm-up (MA/RSI đầu chuỗi) bao nhiêu? Tỷ lệ 3 nhãn tổng thể?


In [ ]:
df = pd.read_csv(PANEL, parse_dates=["date"])
df = df[df["date"] < EDA_END].copy()
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

print(
    f"Rows: {len(df):,} | symbols: {df['symbol'].nunique()} | "
    f"{df['date'].min().date()} → {df['date'].max().date()}"
)
print("\nLabel dist (<2025):")
print(df["label"].value_counts(normalize=True).round(4))
print("\nNaN mỗi feature (warm-up MA/RSI đầu chuỗi mỗi mã):")
print(df[FEATURES].isna().sum())


## 2. Phân phối nhãn theo thời gian (đo regime shift)

**Vì sao quan trọng:** v3 đã phát hiện threshold chọn trên val 2024 KHÔNG transfer sang test 2025.
Nếu tỷ lệ nhãn trôi mạnh theo năm → val phải chọn gần test hơn khi train v4.

**Câu hỏi:** tỷ lệ `di_ngang` có trôi theo năm không? 2024 khác gì 2018–2023? Có năm nào bất thường (2018 khủng hoảng, 2020 COVID, 2022 trái phiếu)?


In [ ]:
df["year"] = df["date"].dt.year

share_y = df.groupby("year")["label"].value_counts(normalize=True).unstack().fillna(0)
ax = share_y[CLASSES].plot(
    kind="bar", stacked=True, figsize=(12, 4), color=["#999", "#d9534f", "#5cb85c"]
)
ax.set_title("Tỷ lệ nhãn theo năm — nhìn độ trôi (regime shift)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

recent = df[df["year"] >= 2022]
share_q = (
    recent.groupby(recent["date"].dt.to_period("Q"))["label"]
    .value_counts(normalize=True)
    .unstack()
)
share_q[CLASSES].plot(figsize=(12, 3), marker="o", color=["#999", "#d9534f", "#5cb85c"])
plt.title("Tỷ lệ nhãn theo quý 2022+ — val 2024 có giống các quý trước không?")
plt.tight_layout()
plt.show()


## 3. Feature hiện có vs nhãn

**Câu hỏi:** feature nào tách lớp rõ nhất (kỳ vọng RSI14 — importance v3)? `ma7`/`ma20`/`close` có redundant (corr ~1) không? `sentiment_agg` có đáng giữ khi importance v3 = 0.0?


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, f in zip(axes.flat, FEATURES):
    sns.boxplot(data=df, x="label", y=f, order=CLASSES, ax=ax, showfliers=False)
    ax.set_title(f)
axes.flat[-1].axis("off")
fig.suptitle("Phân phối feature hiện có theo 3 lớp (bỏ outlier cho dễ nhìn)")
plt.tight_layout()
plt.show()


In [ ]:
corr = df[["close"] + FEATURES].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation — kỳ vọng close~ma7~ma20 gần 1 (redundant, nên thay bằng tỷ lệ)")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.feature_selection import mutual_info_classif

# MI dùng kNN — sample 30k cho nhanh, đủ ổn định để xếp hạng
sub = df.dropna(subset=FEATURES)
samp = sub.sample(min(30_000, len(sub)), random_state=SEED)
mi = mutual_info_classif(samp[FEATURES], samp["label"], random_state=SEED)
mi_now = pd.Series(mi, index=FEATURES).sort_values()
mi_now.plot(kind="barh", figsize=(8, 3.5), title="Mutual information — feature hiện có")
plt.tight_layout()
plt.show()
print(mi_now.sort_values(ascending=False).round(4))


## 4. Volatility candidates (ưu tiên #1 — tính từ `close`, leakage-safe)

Tất cả chỉ dùng close ≤ ngày T (`pct_change`/`rolling` nhìn về quá khứ, group theo mã để không tràn sang mã khác):

| Candidate | Ý nghĩa |
|---|---|
| `ret_1d`, `ret_5d` | momentum ngắn |
| `abs_ret_1d` | biên độ hôm nay |
| `vol_5`, `vol_20` | rolling std của return — chế độ biến động |
| `dist_ma20` | (close−ma20)/ma20 — vị trí tương đối, thay cho ma tuyệt đối |
| `ma_ratio` | ma7/ma20 — trend, thay cho 2 cột redundant |

**Giả thuyết chính:** ngày T volatility cao → T+1 ít khả năng `di_ngang`. Nếu đúng → đây là nhóm feature đập trần.


In [ ]:
g = df.groupby("symbol")
df["ret_1d"] = g["close"].pct_change()
df["ret_5d"] = g["close"].pct_change(5)
df["abs_ret_1d"] = df["ret_1d"].abs()
df["vol_5"] = g["ret_1d"].transform(lambda s: s.rolling(5).std())
df["vol_20"] = g["ret_1d"].transform(lambda s: s.rolling(20).std())
df["dist_ma20"] = (df["close"] - df["ma20"]) / df["ma20"]
df["ma_ratio"] = df["ma7"] / df["ma20"]

CANDS = ["ret_1d", "ret_5d", "abs_ret_1d", "vol_5", "vol_20", "dist_ma20", "ma_ratio"]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, f in zip(axes.flat, CANDS):
    sns.boxplot(data=df, x="label", y=f, order=CLASSES, ax=ax, showfliers=False)
    ax.set_title(f)
axes.flat[-1].axis("off")
fig.suptitle("Volatility/momentum candidates theo 3 lớp")
plt.tight_layout()
plt.show()


In [ ]:
# Kiểm giả thuyết "biên độ trước, hướng sau": P(nhãn T+1 | quintile vol_5 ngày T)
sub = df.dropna(subset=["vol_5"]).copy()
sub["vol_q"] = pd.qcut(sub["vol_5"], 5, labels=["Q1 thấp", "Q2", "Q3", "Q4", "Q5 cao"])
ct = sub.groupby("vol_q", observed=True)["label"].value_counts(normalize=True).unstack()
ax = ct[CLASSES].plot(kind="bar", figsize=(10, 4), color=["#999", "#d9534f", "#5cb85c"])
ax.set_title("P(nhãn T+1 | quintile vol_5 ngày T) — kỳ vọng: vol cao → ít di_ngang")
ax.axhline(df["label"].value_counts(normalize=True)["di_ngang"], ls="--", c="#999", lw=1)
plt.tight_layout()
plt.show()
print(ct[CLASSES].round(3))


## 5. Market context — VNINDEX (vnstock Quote, cache local)

**Finding khảo sát (2026-06-10):** vnstock free KHÔNG có foreign flows lịch sử (`Trading.foreign_trade`
→ `NotImplementedError` mọi nguồn, cả bản 4.0.4). Phần khối ngoại dời sang nấc 1 (tìm nguồn khác).
Ở đây dùng index làm market context.

**Câu hỏi:** ngày VNINDEX biến động mạnh, phân phối nhãn T+1 của các mã đổi thế nào (contagion)?
`rel_return` (mã − index) có tách lớp tốt hơn `ret_1d` thuần không?


In [ ]:
def fetch_history(symbol: str, start: str = "2008-01-01", end: str = "2024-12-31") -> pd.DataFrame:
    """Fetch OHLCV qua vnstock Quote (VCI), cache CSV để không gọi lại API."""
    f = CACHE / f"{symbol}.csv"
    if f.exists():
        return pd.read_csv(f, parse_dates=["time"])
        
    # Thử tìm trong Kaggle inputs nếu chạy trên Kaggle
    if is_kaggle:
        # Tìm kiếm không phân biệt hoa thường dưới /kaggle/input
        potential_files = []
        for p in Path("/kaggle/input").rglob("*"):
            if p.is_file() and p.name.lower() == f"{symbol.lower()}.csv":
                potential_files.append(p)
                
        if potential_files:
            print(f"Found cached file for {symbol} in input dataset: {potential_files[0]}")
            df_cached = pd.read_csv(potential_files[0], parse_dates=["time"])
            df_cached.to_csv(f, index=False)
            return df_cached

    try:
        from vnstock.api.quote import Quote
        out = Quote(symbol=symbol, source="VCI").history(start=start, end=end)
        out["time"] = pd.to_datetime(out["time"])
        out.to_csv(f, index=False)
        return out
    except Exception as e:
        print(f"Lỗi tải dữ liệu qua vnstock cho {symbol}: {e}")
        raise RuntimeError(
            f"Không thể lấy dữ liệu cho {symbol}. Nguyên nhân: API vnstock lỗi hoặc "
            f"Kaggle Cloud IP bị chặn bởi API nguồn của vnstock (VCI).\n"
            f"--> HƯỚNG GIẢI QUYẾT: Hãy chắc chắn bạn đã thêm dataset chứa các file '{symbol}.csv' "
            f"vào Kaggle Notebook này."
        ) from e


vni = fetch_history("VNINDEX").sort_values("time")
vni["index_ret_1d"] = vni["close"].pct_change()
vni["index_ret_5d"] = vni["close"].pct_change(5)
vni["index_vol_5"] = vni["index_ret_1d"].rolling(5).std()
print(f"VNINDEX: {len(vni):,} phiên, {vni['time'].min().date()} → {vni['time'].max().date()}")

idx_feats = vni[["time", "index_ret_1d", "index_ret_5d", "index_vol_5"]].rename(
    columns={"time": "date"}
)
df = df.merge(idx_feats, on="date", how="left")
df["rel_return"] = df["ret_1d"] - df["index_ret_1d"]
MARKET = ["index_ret_1d", "index_ret_5d", "index_vol_5", "rel_return"]
print("NaN sau merge:", df[MARKET].isna().sum().to_dict())

In [ ]:
# P(nhãn T+1 | mức biến động VNINDEX ngày T)
sub = df.dropna(subset=["index_ret_1d"]).copy()
bins = [-np.inf, -0.01, -0.003, 0.003, 0.01, np.inf]
names = ["≤ -1%", "-1%..-0.3%", "±0.3%", "0.3%..1%", "≥ 1%"]
sub["idx_bin"] = pd.cut(sub["index_ret_1d"], bins, labels=names)
ct = sub.groupby("idx_bin", observed=True)["label"].value_counts(normalize=True).unstack()
ax = ct[CLASSES].plot(kind="bar", figsize=(10, 4), color=["#999", "#d9534f", "#5cb85c"])
ax.set_title("P(nhãn T+1 | VNINDEX return ngày T) — index sập/bứt có kéo theo mã không?")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, f in zip(axes, MARKET):
    sns.boxplot(data=df, x="label", y=f, order=CLASSES, ax=ax, showfliers=False)
    ax.set_title(f)
plt.tight_layout()
plt.show()


## 6. Volume thử nghiệm (vài mã đại diện)

Panel chưa có volume. Pull thử 4 mã qua vnstock để xem `rel_volume = volume / MA20(volume)` có đáng
đưa vào builder không (volume phiên T chốt 15:00 < 16:00 → leakage-safe).


In [ ]:
SAMPLE = ["FPT", "HPG", "VNM", "ACB"]
frames = []
for s in SAMPLE:
    d = fetch_history(s)[["time", "volume"]].rename(columns={"time": "date"})
    d["symbol"] = s
    frames.append(d)
vol = pd.concat(frames).sort_values(["symbol", "date"])
vol["rel_volume"] = vol.groupby("symbol")["volume"].transform(
    lambda x: x / x.rolling(20).mean()
)

m = df.merge(vol[["date", "symbol", "rel_volume"]], on=["date", "symbol"], how="inner")
print(f"Join được {len(m):,} hàng ({m['symbol'].nunique()} mã mẫu)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=m, x="label", y="rel_volume", order=CLASSES, ax=ax, showfliers=False)
ax.set_title("rel_volume (4 mã mẫu) theo 3 lớp — volume spike có báo biến động T+1?")
plt.tight_layout()
plt.show()


## 7. Tổng kết — xếp hạng toàn bộ candidates

MI tổng hợp để xếp hạng tương đối. **Nhớ:** đây chỉ là sàng lọc; phán quyết cuối là macro-F1
walk-forward khi train v4 (gate > 0.4018). Mỗi feature được chọn phải có test no-leakage riêng ở nấc builder.


In [ ]:
ALL = FEATURES + CANDS + MARKET
sub = df.dropna(subset=ALL)
samp = sub.sample(min(30_000, len(sub)), random_state=SEED)
mi = mutual_info_classif(samp[ALL], samp["label"], random_state=SEED)
rank = pd.Series(mi, index=ALL).sort_values(ascending=False)

rank.sort_values().plot(kind="barh", figsize=(8, 7), title="MI tổng hợp — cũ vs candidates mới")
plt.tight_layout()
plt.show()
print(rank.round(4))


## 📝 Insight chốt (grill session 2026-06-11)

**Panel EDA:** 86,243 hàng, 30 mã, 2008-03-06 → 2024-12-31. Nhãn <2025: **di_ngang 51.6% / tang 24.7% / giam 23.7%** (baseline ~0.52, khớp v3).

- **Regime shift theo năm/quý:** CÓ trôi rõ. 2008–2009 dị thường (di_ngang chỉ 15–28%, khủng hoảng). 2018 (41%) vs 2019/2023/2024 (57–63%) — biên độ ~20 điểm %. **Quý 2024 di_ngang leo dốc 57%→72% (Q4)** → val 2024 KHÁC các quý trước, xác nhận lại bài học v3 (threshold không transfer val 2024→test 2025).
- **Nhóm volatility — "vol cao → ít di_ngang": ĐÚNG MẠNH** (tín hiệu sạch nhất notebook). Quintile vol_5: Q1 thấp → di_ngang **68.8%**; Q5 cao → **33.0%** (giam 32.3 / tang 34.7). Monotonic hoàn hảo → vol_5 ngày T dự báo **biên độ** T+1. Đúng luận đề "biên độ trước, hướng sau".
- **Index features có contagion KHÔNG → KHÔNG.** ⚠️ **MI 0.194 của index_ret_1d/5d/index_vol_5 là GIẢ.** Kiểm chứng (grill 2026-06-11): MI trên panel 30 mã = 0.194 → trên **1 mã FPT = 0.001** (sụp ~200×); shuffle index trong từng năm → MI rớt về 0.018. Nguyên nhân: index lặp y hệt 30 lần/ngày → kNN nhớ "ngày nào" chứ không học "index→nhãn". Boxplot 3 lớp (cell 15) chồng hoàn toàn, P(nhãn|index return) phẳng. → **LOẠI cả 4 index feature.**
- **rel_volume:** chỉ thử 4 mã mẫu (16,792 hàng) → bằng chứng yếu, KHÔNG đủ để build cho cả 30 mã. **Hoãn** (cân nhắc lại nếu v4 kẹt; volume cần fetch thêm vào panel).
- **Feature redundant BỎ:** `ma7`/`ma20` corr ~1 với close → BỎ, thay bằng `dist_ma20` + `ma_ratio`. `sentiment_agg`/`news_count` MI=0 (stub chết) → BỎ khỏi v4, thêm lại ở M8.

**→ Danh sách feature chốt cho lgbm_v4 (10 giá nội tại + 1 categorical):**
`rsi14, macd, macd_signal` (cũ giữ) + `ret_1d, ret_5d, abs_ret_1d, vol_5, vol_20, dist_ma20, ma_ratio` (mới) + `sector` (static categorical).

**Split (dịch tiến, panel local đã tới 2026-06-09):** train **2010→2024** / val **2025** / test **2026 (H1)**. Cắt 2008–2009 (4.4% data, chế độ dị thường) — đòn bẩy bật lại nếu v4 không nhúc. Threshold + temperature tune trên **toàn val 2025**; báo cáo thêm số 2025-as-reference để so gần-công-bằng với v3 (v3 test trên 2025). Gate: macro-F1 > 0.4018 + giữ vùng gating precision ≥0.50/coverage ≥20% trên test.

**Lưu ý:** notebook EDA chạy panel cũ (→2024); train v4 phải **re-export panel local** (→2026) + builder thêm feature mới trước.
